# Somnotate Testing
Run Somnotate on annotated .mat files in data/hypnose_eeg/somnotate_testing,
then visualize scores against manual annotations and compute agreements.

In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
from utils.testing import (
    prepare_testing_csvs,
    load_manual_vectors_from_csvs,
    load_somnotate_predictions,
    save_somnotate_predictions,
    score_somnotate,
    align_vectors,
    plot_testing_comparison,
    agreement_matrix,
    plot_agreement_matrix,
)
%matplotlib qt


In [ ]:
model_name = "test_model_2"
current_test = "sub_53_comparison"
repo_root = Path.cwd().parent
testing_mat_dir = repo_root / "data" / "hypnose_eeg" / "somnotate_testing" / current_test
model_path = (
    repo_root
    / "data"
    / "hypnose_eeg"
    / "derivatives"
    / "somnotate_training"
    / model_name
    / "model.pickle"
)

print("testing_mat_dir:", testing_mat_dir)
print("model_path:", model_path)

testing_mat_dir: /Users/joschua/repos/harris_lab/eeg_preprocessing/data/hypnose_eeg/somnotate_testing/sub_53_comparison
model_path: /Users/joschua/repos/harris_lab/eeg_preprocessing/data/hypnose_eeg/derivatives/somnotate_training/test_model_2/model.pickle


In [7]:
csv_dir = prepare_testing_csvs(
    testing_mat_dir=testing_mat_dir,
    repo_root=repo_root,
    model_name=model_name,
    test_name=current_test,
)

csv_files = sorted([p for p in csv_dir.glob("*.csv") if not p.name.startswith("._")])
print("CSV files:", len(csv_files))
for path in csv_files:
    print(" -", path.name)

predictions_dir = (
    repo_root
    / "data"
    / "hypnose_eeg"
    / "derivatives"
    / "somnotate_testing"
    / model_name
    / current_test
    / "somnotate_predictions"
)

reference_csv = csv_files[0] if csv_files else None
if reference_csv is None:
    raise ValueError("No CSV files found in testing output")

somnotate_pred_path, _ = save_somnotate_predictions(
    reference_csv,
    model_path,
    predictions_dir,
)
print("Somnotate predictions:", somnotate_pred_path)

Found .mat files: ['/Users/joschua/repos/harris_lab/eeg_preprocessing/data/hypnose_eeg/somnotate_testing/sub_53_comparison/sub-006_ses-01_recording-02_Nam.mat', '/Users/joschua/repos/harris_lab/eeg_preprocessing/data/hypnose_eeg/somnotate_testing/sub_53_comparison/sub-006_ses-01_recording-02_Joschua.mat', '/Users/joschua/repos/harris_lab/eeg_preprocessing/data/hypnose_eeg/somnotate_testing/sub_53_comparison/sub-006_ses-01_recording-02_Volkan.mat']
Processing file: /Users/joschua/repos/harris_lab/eeg_preprocessing/data/hypnose_eeg/somnotate_testing/sub_53_comparison/sub-006_ses-01_recording-02_Nam.mat
EEG1 data extracted successfully.
EEG2 data extracted successfully.
EMG data extracted successfully.
Sleep stage data extracted successfully.
Length of upsampled sleep stages (17981440) does not match length of EEG data (17981441) by 1 samples
Upsampled sleep stages padded with zeros to match length of EEG data
Length of upsampled sleep stages matches length of EEG data after truncation
Sa

In [8]:
# Pick one or more CSVs to compare (each file can contain one or more sleepStage columns).
csv_paths = "all"  # use "all" or a list like [csv_files[0], csv_files[1]]

if csv_paths == "all":
    csv_paths = csv_files

raw_signals, manual_vectors = load_manual_vectors_from_csvs(csv_paths)
somnotate_vec = load_somnotate_predictions(somnotate_pred_path)
somnotate_vec, manual_vectors = align_vectors(somnotate_vec, manual_vectors)

fig, viewer = plot_testing_comparison(
    raw_signals,
    sampling_rate_hz=512,
    somnotate_vec=somnotate_vec,
    manual_vectors=manual_vectors,
 )

In [ ]:
agreement_df = agreement_matrix(somnotate_vec, manual_vectors)
agreement_df

agreement_plot_path = (
    repo_root
    / "data"
    / "hypnose_eeg"
    / "derivatives"
    / "somnotate_testing"
    / model_name
    / current_test
    / "agreement_matrix.png"
)

plot_agreement_matrix(agreement_df, output_path=agreement_plot_path)
print("Saved agreement matrix:", agreement_plot_path)

Saved agreement matrix: /Users/joschua/repos/harris_lab/eeg_preprocessing/data/hypnose_eeg/derivatives/somnotate_testing/test_model_2/sub_53_comparison/agreement_matrix.png
